In [143]:
pdf_path='07 ViPNet Coordinator HW 5. Перечень совместимых трансиверов.pdf'

In [ ]:
import fitz # (pymupdf, found this is better than pypdf for our use case, note: licence is AGPL-3.0, keep that in mind if you want to use any code commercially)
from tqdm.auto import tqdm # for progress bars, requires !pip install tqdm 

def text_formatter(text: str) -> str:
    """Performs minor formatting on text."""
    cleaned_text = text.replace("\n", " ").strip() # note: this might be different for each doc (best to experiment)

    # Other potential text formatting functions can go here
    return cleaned_text

# Open PDF and get lines/pages
# Note: this only focuses on text, rather than images/figures etc
def open_and_read_pdf(pdf_path: str) -> list[dict]:
    """
    Opens a PDF file, reads its text content page by page, and collects statistics.

    Parameters:
        pdf_path (str): The file path to the PDF document to be opened and read.

    Returns:
        list[dict]: A list of dictionaries, each containing the page number
        (adjusted), character count, word count, sentence count, token count, and the extracted text
        for each page.
    """
    doc = fitz.open(pdf_path)  # open a document
    pages_and_texts = []
    for page_number, page in tqdm(enumerate(doc)):  # iterate the document pages
        text = page.get_text()  # get plain text encoded as UTF-8
        text = text_formatter(text)
        pages_and_texts.append({"page_number": page_number,  # adjust page numbers 
                                "page_char_count": len(text),
                                "page_word_count": len(text.split(" ")),
                                "page_sentence_count_raw": len(text.split(". ")),
                                "page_token_count": len(text) / 4,  # 1 token = ~4 chars, see: https://help.openai.com/en/articles/4936856-what-are-tokens-and-how-to-count-them
                                "text": text})
    return pages_and_texts

pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)
pages_and_texts[:2]

6it [00:00, 275.64it/s]


[{'page_number': 0,
  'page_char_count': 1080,
  'page_word_count': 264,
  'page_sentence_count_raw': 2,
  'page_token_count': 270.0,
  'text': 'Copyright (c) InfoTeCS. All Rights Reserved     Аппаратные платформы HW100 N1, N2, N3  Модель  Тип  Длина волны, нм  Интерфейс  Тип кабеля  Расстояние, км  Avago  AFBR-5710PZ  SFP  850  LC duplex  MM  0,5  Allied Telesis  AT-SPEX  SFP  1310  LC duplex  MM  2  FINISAR  FTLX8574D3BCV  SFP/SFP+  850  LC duplex  MM  0,4  FTLX8519P3BNL  SFP  850  LC duplex  MM  0,3  FTLF1318P3BTL  SFP  1310  LC duplex  SM  10  MlaxLink  ML-P10G-03DFM-85LD  SFP+  850  LC duplex  MM  0,3  ML-SG-01UTP-SGMRJ  SFP  -  RJ-45  UTP  0,1  Smartoptics  SO-SFP-10GE-T  SFP+  -  RJ-45  -  0,03  Аппаратная платформа HW100 Q1, Q2  Модель  Тип  Длина волны, нм  Интерфейс  Тип кабеля  Расстояние, км  Avago  AFBR-5710PZ  SFP  850  LC duplex  MM  0,5  FINISAR  FTLX8574D3BCV  SFP/SFP+  850  LC duplex  MM  0,4  SNR  SNR-SFP-W35-3-LC  SFP  1310  SC simplex  SM  3  Аппаратная платформа H

In [145]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)
df.head()

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text
0,0,1080,264,2,270.00,Copyright (c) InfoTeCS. All Rights Reserved ...
1,1,1140,309,1,285.00,"Модель Тип Длина волны, нм Интерфейс Тип к..."
2,2,1055,291,1,263.75,"Аппаратные платформы HW2000 Q4, Q5 Модель Ти..."
3,3,1150,313,1,287.50,Аппаратная платформа HW5000 Q1 Модель Тип Д...
4,4,1059,299,1,264.75,"Модель Тип Длина волны, нм Интерфейс Тип к..."


In [146]:
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count
count,6.00,6.00,6.00,6.00,6.00
mean,2.50,1004.00,258.00,1.83,251.00
std,1.87,230.87,92.77,1.60,57.72
min,0.00,540.00,72.00,1.00,135.00
25%,1.25,1056.00,270.75,1.00,264.00
50%,2.50,1069.50,295.00,1.00,267.38
75%,3.75,1125.00,306.50,1.75,281.25
max,5.00,1150.00,313.00,5.00,287.50


In [ ]:
from spacy.lang.ru import Russian # see https://spacy.io/usage for install instructions

nlp = Russian()

# Add a sentencizer pipeline, see https://spacy.io/api/sentencizer/ 
nlp.add_pipe("sentencizer")

In [149]:
for item in tqdm(pages_and_texts):
    item["sentences"] = list(nlp(item["text"]).sents)
    
    # Make sure all sentences are strings
    item["sentences"] = [str(sentence) for sentence in item["sentences"]]
    
    # Count the sentences 
    item["page_sentence_count_spacy"] = len(item["sentences"])

100%|██████████| 6/6 [00:00<00:00, 725.89it/s]


In [150]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy
count,6.00,6.00,6.00,6.00,6.00,6.00
mean,2.50,1004.00,258.00,1.83,251.00,1.50
std,1.87,230.87,92.77,1.60,57.72,0.84
min,0.00,540.00,72.00,1.00,135.00,1.00
25%,1.25,1056.00,270.75,1.00,264.00,1.00
50%,2.50,1069.50,295.00,1.00,267.38,1.00
75%,3.75,1125.00,306.50,1.75,281.25,1.75
max,5.00,1150.00,313.00,5.00,287.50,3.00


In [151]:
# Define split size to turn groups of sentences into chunks
num_sentence_chunk_size = 10 

# Create a function that recursively splits a list into desired sizes
def split_list(input_list: list, 
               slice_size: int) -> list[list[str]]:
    """
    Splits the input_list into sublists of size slice_size (or as close as possible).

    For example, a list of 17 sentences would be split into two lists of [[10], [7]]
    """
    return [input_list[i:i + slice_size] for i in range(0, len(input_list), slice_size)]

# Loop through pages and texts and split sentences into chunks
for item in tqdm(pages_and_texts):
    item["sentence_chunks"] = split_list(input_list=item["sentences"],
                                         slice_size=num_sentence_chunk_size)
    item["num_chunks"] = len(item["sentence_chunks"])

100%|██████████| 6/6 [00:00<00:00, 83055.52it/s]


In [152]:
# Create a DataFrame to get stats
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy,num_chunks
count,6.00,6.00,6.00,6.00,6.00,6.00,6.0
mean,2.50,1004.00,258.00,1.83,251.00,1.50,1.0
std,1.87,230.87,92.77,1.60,57.72,0.84,0.0
min,0.00,540.00,72.00,1.00,135.00,1.00,1.0
25%,1.25,1056.00,270.75,1.00,264.00,1.00,1.0
50%,2.50,1069.50,295.00,1.00,267.38,1.00,1.0
75%,3.75,1125.00,306.50,1.75,281.25,1.75,1.0
max,5.00,1150.00,313.00,5.00,287.50,3.00,1.0


In [ ]:
import re

pages_and_chunks = []
for item in tqdm(pages_and_texts):
    for sentence_chunk in item["sentence_chunks"]:
        chunk_dict = {}
        chunk_dict["page_number"] = item["page_number"]
        
        joined_sentence_chunk = "".join(sentence_chunk).replace("  ", " ").strip()
        joined_sentence_chunk = re.sub(r'\.([A-Z])', r'. \1', joined_sentence_chunk)
        chunk_dict["sentence_chunk"] = joined_sentence_chunk
        chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
        chunk_dict["chunk_word_count"] = len([word for word in joined_sentence_chunk.split(" ")])
        chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4 # 1 token = ~4 characters
        
        pages_and_chunks.append(chunk_dict)

len(pages_and_chunks)

100%|██████████| 6/6 [00:00<00:00, 29026.33it/s]


6

In [154]:
# Get stats about our chunks
df = pd.DataFrame(pages_and_chunks)
df.describe().round(2)

,page_number,chunk_char_count,chunk_word_count,chunk_token_count
count,6.00,6.00,6.00,6.00
mean,2.50,895.00,149.00,223.75
std,1.87,181.41,41.80,45.35
min,0.00,533.00,65.00,133.25
25%,1.25,923.25,157.75,230.81
50%,2.50,948.50,161.50,237.12
75%,3.75,997.00,171.25,249.25
max,5.00,1012.00,175.00,253.00


In [106]:
df = df[df['page_number'] > 0]

In [155]:
# Get stats about our chunks
df.describe().round(2)

,page_number,chunk_char_count,chunk_word_count,chunk_token_count
count,6.00,6.00,6.00,6.00
mean,2.50,895.00,149.00,223.75
std,1.87,181.41,41.80,45.35
min,0.00,533.00,65.00,133.25
25%,1.25,923.25,157.75,230.81
50%,2.50,948.50,161.50,237.12
75%,3.75,997.00,171.25,249.25
max,5.00,1012.00,175.00,253.00


In [156]:
df

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count
0,0,Copyright (c) InfoTeCS. All Rights Reserved ...,973,157,243.25
1,1,"Модель Тип Длина волны, нм Интерфейс Тип кабел...",1005,174,251.25
2,2,"Аппаратные платформы HW2000 Q4, Q5 Модель Тип ...",924,160,231.00
3,3,Аппаратная платформа HW5000 Q1 Модель Тип Длин...,1012,175,253.00
4,4,"Модель Тип Длина волны, нм Интерфейс Тип кабел...",923,163,230.75
5,5,"АО «ИнфоТеКС», 127083, Москва, улица Мишина, д...",533,65,133.25


In [15]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer(model_name_or_path="ai-forever/ru-en-RoSBERTa", 
                                      device="cuda") # choose the device to load the model to (note: GPU will often be *much* faster than CPU)

2026-01-28 06:57:09.747453: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [157]:
pages_and_chunks = df.sentence_chunk

In [158]:
pages_and_chunks.tolist()

['Copyright (c) InfoTeCS. All Rights Reserved   Аппаратные платформы HW100 N1, N2, N3 Модель Тип Длина волны, нм Интерфейс Тип кабеля Расстояние, км Avago AFBR-5710PZ SFP 850 LC duplex MM 0,5 Allied Telesis AT-SPEX SFP 1310 LC duplex MM 2 FINISAR FTLX8574D3BCV SFP/SFP+ 850 LC duplex MM 0,4 FTLX8519P3BNL SFP 850 LC duplex MM 0,3 FTLF1318P3BTL SFP 1310 LC duplex SM 10 MlaxLink ML-P10G-03DFM-85LD SFP+ 850 LC duplex MM 0,3 ML-SG-01UTP-SGMRJ SFP - RJ-45 UTP 0,1 Smartoptics SO-SFP-10GE-T SFP+ - RJ-45 - 0,03 Аппаратная платформа HW100 Q1, Q2 Модель Тип Длина волны, нм Интерфейс Тип кабеля Расстояние, км Avago AFBR-5710PZ SFP 850 LC duplex MM 0,5 FINISAR FTLX8574D3BCV SFP/SFP+ 850 LC duplex MM 0,4 SNR SNR-SFP-W35-3-LC SFP 1310 SC simplex SM 3 Аппаратная платформа HW1000 Q6 Модель Тип Длина волны, нм Интерфейс Тип кабеля Расстояние, км Avago AFBR-5710PZ SFP 850 LC duplex MM 0,5 Allied Telesis ViPNet Coordinator HW 5 Перечень совместимых трансиверов Версия продукта 5.3.2',
 'Модель Тип Длина вол

In [159]:
text_chunk_embeddings=[]
for i in pages_and_chunks.tolist():
    text_chunk_embeddings.append(embedding_model.encode(i,
                                               batch_size=32, # you can use different batch sizes here for speed/performance, I found 32 works well for this use case
                                               convert_to_tensor=True))

In [160]:
text_chunk_embeddings

[tensor([ 0.0042, -0.0040,  0.0231,  ...,  0.0195, -0.0190, -0.0074],
        device='cuda:0'),
 tensor([-0.0147,  0.0232,  0.0069,  ...,  0.0107, -0.0097, -0.0114],
        device='cuda:0'),
 tensor([ 0.0024,  0.0181,  0.0194,  ...,  0.0186, -0.0066, -0.0167],
        device='cuda:0'),
 tensor([ 0.0007,  0.0152,  0.0219,  ...,  0.0185, -0.0009, -0.0157],
        device='cuda:0'),
 tensor([-2.3185e-03,  1.3453e-02,  2.1185e-02,  ...,  1.7644e-02,
         -6.6569e-05, -1.6092e-02], device='cuda:0'),
 tensor([-0.0109, -0.0332,  0.0499,  ...,  0.0034,  0.0307, -0.0284],
        device='cuda:0')]

In [161]:
text_chunk_embeddings_list = []
for i in text_chunk_embeddings:
    text_chunk_embeddings_list.append(i.tolist())

In [162]:
text_chunk_embeddings_list

[[0.004246750380843878,
  -0.003971358295530081,
  0.02313687652349472,
  0.01397034339606762,
  -0.006961638107895851,
  0.013670289888978004,
  0.026896491646766663,
  0.0075852712616324425,
  0.022998834028840065,
  0.008264067582786083,
  0.012624431401491165,
  -0.0005829408764839172,
  0.013871158473193645,
  -0.03723147511482239,
  -0.020602280274033546,
  0.06948986649513245,
  -0.03298730403184891,
  -0.033339500427246094,
  -0.012561855837702751,
  -0.021445589140057564,
  -0.0014110993361100554,
  0.03477207571268082,
  0.01993366703391075,
  0.03633282706141472,
  0.01824379526078701,
  0.01754799857735634,
  -0.012471224181354046,
  -0.033341292291879654,
  -0.0024051533546298742,
  0.015582953579723835,
  0.001729471143335104,
  0.014078963547945023,
  0.00159259676001966,
  -0.0017826429102569818,
  0.010386661626398563,
  0.0149122579023242,
  -0.02265290915966034,
  0.003797390731051564,
  -0.028591757640242577,
  -0.011390988714993,
  -0.06350996345281601,
  -0.018269

In [163]:
df['embedding'] = text_chunk_embeddings_list
df

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding
0,0,Copyright (c) InfoTeCS. All Rights Reserved ...,973,157,243.25,"[0.004246750380843878, -0.003971358295530081, ..."
1,1,"Модель Тип Длина волны, нм Интерфейс Тип кабел...",1005,174,251.25,"[-0.014727634377777576, 0.023174844682216644, ..."
2,2,"Аппаратные платформы HW2000 Q4, Q5 Модель Тип ...",924,160,231.00,"[0.0024295246694236994, 0.018062317743897438, ..."
3,3,Аппаратная платформа HW5000 Q1 Модель Тип Длин...,1012,175,253.00,"[0.0006676093325950205, 0.015232657082378864, ..."
4,4,"Модель Тип Длина волны, нм Интерфейс Тип кабел...",923,163,230.75,"[-0.0023184509482234716, 0.013453398831188679,..."
5,5,"АО «ИнфоТеКС», 127083, Москва, улица Мишина, д...",533,65,133.25,"[-0.010863716714084148, -0.03315876051783562, ..."


In [164]:
# Save embeddings to file
text_chunks_and_embeddings_df = pd.DataFrame(df)
embeddings_df_save_path = "07_embeddings_df.csv"
text_chunks_and_embeddings_df.to_csv(embeddings_df_save_path, index=False)

In [165]:
df7 =df

In [167]:
df1['product']='подготовка'
df1

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding,product
16,1,ViPNet Coordinator HW 5.Подготовка к работе | ...,1274,159,318.50,"[0.0060768756084144115, -0.042224615812301636,...",подготовка
17,1,NAT позволяет решить две задачи:  Подключение...,930,112,232.50,"[-0.007902384735643864, -0.02230028621852398, ...",подготовка
18,2,ViPNet Coordinator HW 5.Подготовка к работе | ...,234,35,58.50,"[0.008719977922737598, -0.03479960188269615, -...",подготовка
19,3,ViPNet Coordinator HW 5.Подготовка к работе | ...,1222,149,305.50,"[-0.022762130945920944, -0.0357675738632679, 0...",подготовка
20,3,Событие срабатывания правила IPS регистрируетс...,314,43,78.50,"[-0.017361363396048546, 0.0026617932599037886,...",подготовка
...,...,...,...,...,...,...,...
150,82,В ViPNet Coordinator HW версии 5.0 и выше испо...,972,125,243.00,"[-0.010205193422734737, -0.04487336426973343, ...",подготовка
151,83,ViPNet Coordinator HW 5.Подготовка к работе | ...,1153,138,288.25,"[-0.00508173368871212, -0.054079338908195496, ...",подготовка
152,83,"Сеть ViPNet Логическая сеть, организованная с ...",484,59,121.00,"[0.011589154601097107, -0.03198278695344925, 0...",подготовка
153,84,ViPNet Coordinator HW 5.Подготовка к работе | ...,1197,140,299.25,"[-0.010457828640937805, -0.030137909576296806,...",подготовка


In [169]:
df2['product']='cli'
df2

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding,product
21,1,ViPNet Coordinator HW 5.Настройка с помощью ко...,1501,194,375.25,"[0.005142953246831894, -0.011179185472428799, ...",cli
22,1,Контекстная справка вызывается с помощью симво...,51,7,12.75,"[-0.0031007288489490747, 0.03535120561718941, ...",cli
23,2,ViPNet Coordinator HW 5.Настройка с помощью ко...,1501,228,375.25,"[-0.013882667757570744, -0.021430110558867455,...",cli
24,3,ViPNet Coordinator HW 5.Настройка с помощью ко...,783,115,195.75,"[0.0022246644366532564, 0.007002372294664383, ...",cli
25,4,ViPNet Coordinator HW 5.Настройка с помощью ко...,662,104,165.50,"[-0.003028304548934102, -0.003923141397535801,...",cli
...,...,...,...,...,...,...,...
709,459,ViPNet Coordinator HW 5.Настройка с помощью ко...,1381,167,345.25,"[-0.015024119056761265, -0.03959723562002182, ...",cli
710,459,В сети ViPNet-координатор выполняет серверные ...,416,43,104.00,"[0.008524478413164616, -0.0642639696598053, 0....",cli
711,460,ViPNet Coordinator HW 5.Настройка с помощью ко...,1102,135,275.50,"[0.0068259150721132755, -0.04547770693898201, ...",cli
712,460,"Сеть ViPNet имеет наложенную маршрутизацию, об...",466,54,116.50,"[-0.004009568132460117, -0.026541519910097122,...",cli


In [170]:
df3['product']='web'
df3

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding,product
15,1,ViPNet Coordinator HW 5.Настройка с помощью ве...,1247,158,311.75,"[-0.005449722986668348, -0.018273696303367615,...",web
16,1,o Для централизованной учётной записи нажмите ...,128,20,32.00,"[-0.05698039382696152, -0.0065096342004835606,...",web
17,2,ViPNet Coordinator HW 5.Настройка с помощью ве...,789,104,197.25,"[-0.01740676537156105, -0.021096395328640938, ...",web
18,2,3.4 Нажмите Войти. Примечание. USB-токен долж...,477,63,119.25,"[-0.023216310888528824, 0.05547713860869408, -...",web
19,3,ViPNet Coordinator HW 5.Настройка с помощью ве...,996,124,249.00,"[0.008452839218080044, -0.0075549413450062275,...",web
...,...,...,...,...,...,...,...
524,333,ViPNet Coordinator HW 5.Настройка с помощью ве...,1092,130,273.00,"[-0.0026286784559488297, -0.03871087729930878,...",web
525,333,Открытый трафик Поток незашифрованных IP-пакет...,393,52,98.25,"[-0.0017361302161589265, -0.04960127919912338,...",web
526,334,ViPNet Coordinator HW 5.Настройка с помощью ве...,1041,119,260.25,"[-0.006632652599364519, -0.042085472494363785,...",web
527,334,"Туннелируемый узел Узел, на котором не установ...",615,70,153.75,"[-0.014668844640254974, -0.06421870738267899, ...",web


In [171]:
df4['product']='справочник'
df4

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding,product
25,1,ViPNet Coordinator HW 5.Справочник команд и ко...,1076,134,269.00,"[-0.02569282427430153, -0.011641775257885456, ...",справочник
26,2,ViPNet Coordinator HW 5.Справочник команд и ко...,834,104,208.50,"[-0.01287016086280346, -0.0021392342168837786,...",справочник
27,2,Особенности использования  Значение счётчика ...,250,34,62.50,"[0.007124020252376795, 0.024593856185674667, -...",справочник
28,3,ViPNet Coordinator HW 5.Справочник команд и ко...,759,90,189.75,"[-0.009648045524954796, 0.01273731142282486, 0...",справочник
29,3,"В других сессиях пользователя, выполнившего ко...",668,100,167.00,"[-0.016790637746453285, 0.03096672147512436, -...",справочник
...,...,...,...,...,...,...,...
774,512,ViPNet Coordinator HW 5.Справочник команд и ко...,1188,135,297.00,"[0.003303920617327094, -0.034523844718933105, ...",справочник
775,512,Для организации взаимодействия между узлами ра...,446,50,111.50,"[0.006700849160552025, -0.011516018770635128, ...",справочник
776,513,ViPNet Coordinator HW 5.Справочник команд и ко...,980,122,245.00,"[0.00471540866419673, -0.05311654135584831, 0....",справочник
777,513,"Транспортная квитанция Файл, оповещающий отпра...",615,66,153.75,"[-0.027645885944366455, -0.015465979464352131,...",справочник


In [172]:
df6['product']='история'
df6

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding,product
0,0,Copyright (c) InfoTeCS. All Rights Reserved ...,1565,206,391.25,"[0.015076713636517525, -0.02475527487695217, 0...",история
1,1, Новое устройство аутентификации Рутокен ЭЦП ...,1686,206,421.50,"[-0.011273158714175224, 0.022998301312327385, ...",история
2,1,o В раздел Состояние системы веб-интерфейса до...,1033,142,258.25,"[0.0031716336961835623, -0.01651792600750923, ...",история
3,2, Изменение в настройке экспорта событий журна...,1504,202,376.00,"[-0.02878248319029808, -0.02799868956208229, 0...",история
4,3, iplir option set max-connections перенесены ...,1132,151,283.00,"[0.004034334793686867, -0.01688470132648945, 0...",история
5,3, Счетчики срабатывания правил межсетевого экр...,1105,136,276.25,"[0.00948849506676197, -0.006793113891035318, 0...",история
6,4, Новый журнал DNS-запросов Добавлен журнал DN...,1588,212,397.00,"[-0.000603150692768395, 0.008206577971577644, ...",история
7,4, Настройка туннелирования локальных адресов Д...,721,92,180.25,"[-0.02508709765970707, -0.018153706565499306, ...",история
8,5,запуска новых приложений.Также объем доступной...,1088,154,272.00,"[-3.625449608080089e-05, -0.035202719271183014...",история
9,5,o VMware vSphere ESXi и Workstation Pro. o Mic...,725,96,181.25,"[0.012960095889866352, 0.006272259168326855, 0...",история


In [173]:
df7['product']='совместимость'
df7

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding,product
0,0,Copyright (c) InfoTeCS. All Rights Reserved ...,973,157,243.25,"[0.004246750380843878, -0.003971358295530081, ...",совместимость
1,1,"Модель Тип Длина волны, нм Интерфейс Тип кабел...",1005,174,251.25,"[-0.014727634377777576, 0.023174844682216644, ...",совместимость
2,2,"Аппаратные платформы HW2000 Q4, Q5 Модель Тип ...",924,160,231.00,"[0.0024295246694236994, 0.018062317743897438, ...",совместимость
3,3,Аппаратная платформа HW5000 Q1 Модель Тип Длин...,1012,175,253.00,"[0.0006676093325950205, 0.015232657082378864, ...",совместимость
4,4,"Модель Тип Длина волны, нм Интерфейс Тип кабел...",923,163,230.75,"[-0.0023184509482234716, 0.013453398831188679,...",совместимость
5,5,"АО «ИнфоТеКС», 127083, Москва, улица Мишина, д...",533,65,133.25,"[-0.010863716714084148, -0.03315876051783562, ...",совместимость


In [175]:
dfi=pd.concat([df1,df2,df3,df4,df6,df7])
dfi

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding,product
16,1,ViPNet Coordinator HW 5.Подготовка к работе | ...,1274,159,318.50,"[0.0060768756084144115, -0.042224615812301636,...",подготовка
17,1,NAT позволяет решить две задачи:  Подключение...,930,112,232.50,"[-0.007902384735643864, -0.02230028621852398, ...",подготовка
18,2,ViPNet Coordinator HW 5.Подготовка к работе | ...,234,35,58.50,"[0.008719977922737598, -0.03479960188269615, -...",подготовка
19,3,ViPNet Coordinator HW 5.Подготовка к работе | ...,1222,149,305.50,"[-0.022762130945920944, -0.0357675738632679, 0...",подготовка
20,3,Событие срабатывания правила IPS регистрируетс...,314,43,78.50,"[-0.017361363396048546, 0.0026617932599037886,...",подготовка
...,...,...,...,...,...,...,...
1,1,"Модель Тип Длина волны, нм Интерфейс Тип кабел...",1005,174,251.25,"[-0.014727634377777576, 0.023174844682216644, ...",совместимость
2,2,"Аппаратные платформы HW2000 Q4, Q5 Модель Тип ...",924,160,231.00,"[0.0024295246694236994, 0.018062317743897438, ...",совместимость
3,3,Аппаратная платформа HW5000 Q1 Модель Тип Длин...,1012,175,253.00,"[0.0006676093325950205, 0.015232657082378864, ...",совместимость
4,4,"Модель Тип Длина волны, нм Интерфейс Тип кабел...",923,163,230.75,"[-0.0023184509482234716, 0.013453398831188679,...",совместимость


In [176]:
# Save embeddings to file
text_chunks_and_embeddings_df = pd.DataFrame(dfi)
embeddings_df_save_path = "all_embeddings_df.csv"
text_chunks_and_embeddings_df.to_csv(embeddings_df_save_path, index=False)

In [177]:
import random

import torch
import numpy as np 
import pandas as pd

device = "cuda" 

# Import texts and embedding df
text_chunks_and_embedding_df = pd.read_csv("all_embeddings_df.csv")

text_chunks_and_embedding_df.head()

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding,product
0,1,ViPNet Coordinator HW 5.Подготовка к работе | ...,1274,159,318.5,"[0.0060768756084144115, -0.042224615812301636,...",подготовка
1,1,NAT позволяет решить две задачи:  Подключение...,930,112,232.5,"[-0.007902384735643864, -0.02230028621852398, ...",подготовка
2,2,ViPNet Coordinator HW 5.Подготовка к работе | ...,234,35,58.5,"[0.008719977922737598, -0.03479960188269615, -...",подготовка
3,3,ViPNet Coordinator HW 5.Подготовка к работе | ...,1222,149,305.5,"[-0.022762130945920944, -0.0357675738632679, 0...",подготовка
4,3,Событие срабатывания правила IPS регистрируетс...,314,43,78.5,"[-0.017361363396048546, 0.0026617932599037886,...",подготовка


In [178]:
a = text_chunks_and_embedding_df['embedding'][0]
a

'[0.0060768756084144115, -0.042224615812301636, 0.03113720938563347, -0.0034974946174770594, 0.030625909566879272, 0.04431794956326485, 0.04278827831149101, -0.03131306171417236, 0.004370264243334532, -0.0018111745594069362, 0.00297339609824121, -0.006709175184369087, -0.029058067128062248, -0.012864574790000916, 0.003598205279558897, 0.0717412605881691, -0.019464312121272087, -0.0007522632367908955, -0.0032086626160889864, 0.01624186895787716, 0.0008060901891440153, 0.009114071726799011, 0.0480445958673954, 0.02245764434337616, 0.034263525158166885, 0.04395190626382828, -0.034268900752067566, 0.0030069672502577305, 0.005287116393446922, -0.02772851102054119, 0.02152939699590206, -0.009112270548939705, -0.00979863665997982, -0.02937152236700058, 0.04139349237084389, 0.031725164502859116, -0.040957432240247726, 0.03895822912454605, -0.01946321502327919, -0.02197045274078846, -0.016633104532957077, -0.02320767380297184, -0.01998130977153778, -0.006218431517481804, -0.008803882636129856, 

In [179]:
b=np.fromstring(a.strip("[]"), sep=", ")
b

array([ 0.00607688, -0.04222462,  0.03113721, ..., -0.01416949,
        0.02664318,  0.01222325], shape=(1024,))